# Данный ноутбук собирает все модели и логику их работы воедино, а так же создает некий интерфейс для удобной работы с моделью, все это сохраняется в файл Full_Model_CV.py, который в дальнейшем может ипользоваться самостоятельно, но будет требовать все необходимые зависимости (сохраненные модели (с весами), а также корректно прописанные пути к снимкам).

In [1]:
%%writefile Full_Model_CV.py

# импортируем модели в ансамбле и дополнительные библиотек
import DenseNet_model
import EfficientNetV2_S_model
import swin_tiny_model

import numpy as np
import pandas as pd

# импортируем мета модель
from meta_model import load_meta_model
predictor = load_meta_model('full_meta_model.pth')

predictor.thresholds['Hernia'] = 0.005 # меняем порог отсечения для класса hernia

# функция для получения истинных заболеваний (если такие данные имеются)
def get_real_diseases(image_path):
    # определяем список с заболеваниями в нужном порядке
    diseases= ['Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion','Emphysema','Fibrosis',
               'Hernia','Infiltration','Mass','No Finding','Nodule','Pleural_Thickening','Pneumonia','Pneumothorax']
    
    train_data = pd.read_csv('3\\train_data.csv')
    teas_data = pd.read_csv('3\\test_data.csv')

    conc = pd.concat([train_data, teas_data], ignore_index=True)
    #print(conc)

    image_ind = image_path[-16:]
    #print(image_ind)

    find_row = conc[conc['Image Index'] == image_ind]
    #print(find_row)
    if find_row.empty:
        return "обьект не найден"

    find_row = find_row.iloc[:, -15:]
    find_row = np.array(find_row)
    #print(find_row)

    real_diseases = []
    for i in range(len(diseases)):
        if find_row[0][i] == 1:
            real_diseases.append(diseases[i])    

    #print(type(find_row))
    return real_diseases



# функция для получения предсказания одного наблюдения
def get_prediction(image_path):
    # получаем логиты с каждой модели в ансамбле
    dn_logits = DenseNet_model.get_logits_from_image(image_path)
    en_logits = EfficientNetV2_S_model.get_logits_from_image(image_path)
    st_logits = swin_tiny_model.get_logits_from_image(image_path)

    # соединяем все логиты в 1 массив
    all_logits = np.concatenate([dn_logits, en_logits, st_logits], axis = 1)

    # прогоняем логиты через мета модель
    result = predictor.predict(all_logits)

    # получаем ответ в формате названия болезней
    predicted_diseases = []
    for i in range(15):
        if result['binary_predictions'][0][i] == 1:
            predicted_diseases.append(result['diseases'][i])

    return [predicted_diseases, result]
    
# функция для получения предсказаний множества наблюдений
def get_predictions_batch(image_path):
    # получаем логиты с каждой модели в ансамбле
    dn_logits = DenseNet_model.get_logits_batch(image_path)
    en_logits = EfficientNetV2_S_model.get_logits_batch(image_path)
    st_logits = swin_tiny_model.get_logits_batch(image_path)

    # соединяем все логиты в 1 массив
    all_logits = np.concatenate([dn_logits, en_logits, st_logits], axis = 1)

    # прогоняем логиты через мета модель
    result = predictor.predict(all_logits)

    # получаем ответ в формате названия болезней
    predicted_diseases = []

    for row in range(len(result['binary_predictions'])):
        row_diseases = []
        curr_row = result['binary_predictions'][row]
        for dis in range(15):
            if curr_row[dis] == 1:
                row_diseases.append(result['diseases'][dis])
        
        predicted_diseases.append(row_diseases)
    return [predicted_diseases, result]

Writing Full_Model_CV.py


In [183]:
aa = get_predictions_batch(["3\\images_001\\images\\00000787_000.png",
                       "3\\images_001\\images\\00000794_001.png"])

Обработано 2/2 изображений


In [184]:
aa

[[['No Finding'], ['No Finding']],
 {'probabilities': array([[1.12640470e-01, 4.56852354e-02, 4.35739048e-02, 4.86559719e-02,
          4.69975397e-02, 1.99414976e-03, 2.98541808e-03, 1.16667734e-03,
          1.60238951e-01, 2.58246139e-02, 6.27444744e-01, 2.62302123e-02,
          6.98689278e-03, 1.37172155e-02, 9.87785961e-03],
         [5.02864551e-03, 2.09811307e-03, 3.99317872e-03, 2.96362006e-04,
          4.55261394e-03, 7.71047780e-04, 6.86452584e-03, 7.21589749e-05,
          1.37247875e-01, 5.04614646e-03, 8.08235466e-01, 2.20334139e-02,
          6.18914422e-03, 3.26743349e-03, 5.22031914e-03]], dtype=float32),
  'binary_predictions': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]],
        dtype=float32),
  'diseases': ['Atelectasis',
   'Cardiomegaly',
   'Consolidation',
   'Edema',
   'Effusion',
   'Emphysema',
   'Fibrosis',
   'Hernia',
   'Infiltration',
   'Mass',
   'No Fin

In [185]:
real = get_real_diseases("3\\images_001\\images\\00000787_000.png")

In [186]:
real

['No Finding']

In [187]:
real1 = get_real_diseases("3\\images_001\\images\\00000794_001.png")

In [188]:
real1

['No Finding']

In [161]:
a = get_prediction('F:\\диплом\\3\\images_001\\images\\00000787_000.png')

In [162]:
a

[['No Finding'],
 {'probabilities': array([[0.11266601, 0.04566075, 0.04356298, 0.04862814, 0.04697372,
          0.0019946 , 0.00298588, 0.00116714, 0.16021165, 0.02582512,
          0.6274852 , 0.026232  , 0.00698696, 0.0137146 , 0.00987906]],
        dtype=float32),
  'binary_predictions': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]],
        dtype=float32),
  'diseases': ['Atelectasis',
   'Cardiomegaly',
   'Consolidation',
   'Edema',
   'Effusion',
   'Emphysema',
   'Fibrosis',
   'Hernia',
   'Infiltration',
   'Mass',
   'No Finding',
   'Nodule',
   'Pleural_Thickening',
   'Pneumonia',
   'Pneumothorax']}]

In [163]:
b = get_real_diseases('F:\\диплом\\3\\images_001\\images\\00000787_000.png')

In [164]:
b

['No Finding']

In [19]:
answ = get_prediction('F:\\диплом\\3\\images_001\\images\\00000001_000.png')

In [20]:
answ

[['Cardiomegaly', 'Hernia', 'No Finding'],
 {'probabilities': array([[0.05479698, 0.24759997, 0.02230356, 0.00268609, 0.08798654,
          0.0193821 , 0.05985972, 0.03361417, 0.1874583 , 0.02828282,
          0.46745786, 0.07548428, 0.03353712, 0.01920267, 0.00239115]],
        dtype=float32),
  'binary_predictions': array([[0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0.]],
        dtype=float32),
  'diseases': ['Atelectasis',
   'Cardiomegaly',
   'Consolidation',
   'Edema',
   'Effusion',
   'Emphysema',
   'Fibrosis',
   'Hernia',
   'Infiltration',
   'Mass',
   'No Finding',
   'Nodule',
   'Pleural_Thickening',
   'Pneumonia',
   'Pneumothorax']}]

In [21]:
train_data = pd.read_csv('3\\train_data.csv')
teas_data = pd.read_csv('3\\test_data.csv')

In [22]:
train_data.shape

(95302, 28)

In [25]:
conc = pd.concat([train_data, teas_data], ignore_index=True)

In [26]:
conc.shape

(112120, 28)

In [28]:
pd.set_option('display.max_columns', None)
conc

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,full_path,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
0,00002137_000.png,Infiltration,0,2137,25,F,PA,2048,2500,0.168000,0.168000,NaN,3\images_002\images\00002137_000.png,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,00029362_001.png,No Finding,1,29362,66,M,PA,2021,2020,0.194311,0.194311,NaN,3\images_012\images\00029362_001.png,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,00003110_000.png,No Finding,0,3110,25,M,PA,2500,2048,0.171000,0.171000,NaN,3\images_002\images\00003110_000.png,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,00013585_000.png,Hernia|Infiltration,0,13585,57,M,PA,2990,2991,0.143000,0.143000,NaN,3\images_006\images\00013585_000.png,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0
4,00019658_013.png,No Finding,13,19658,65,F,AP,3056,2544,0.139000,0.139000,NaN,3\images_009\images\00019658_013.png,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112115,00017489_000.png,No Finding,0,17489,53,M,PA,2586,2991,0.143000,0.143000,NaN,3\images_008\images\00017489_000.png,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
112116,00020773_009.png,Atelectasis|Effusion,9,20773,20,M,PA,2992,2991,0.143000,0.143000,NaN,3\images_009\images\00020773_009.png,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
112117,00028509_031.png,No Finding,31,28509,64,M,AP,3056,2544,0.139000,0.139000,NaN,3\images_012\images\00028509_031.png,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
112118,00029950_000.png,Infiltration,0,29950,36,F,PA,1624,2021,0.194311,0.194311,NaN,3\images_012\images\00029950_000.png,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


In [84]:
def get_real_diseases(image_path):
    # определяем список с заболеваниями в нужном порядке
    diseases= ['Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion','Emphysema','Fibrosis',
               'Hernia','Infiltration','Mass','No Finding','Nodule','Pleural_Thickening','Pneumonia','Pneumothorax']
    
    train_data = pd.read_csv('3\\train_data.csv')
    teas_data = pd.read_csv('3\\test_data.csv')

    conc = pd.concat([train_data, teas_data], ignore_index=True)
    #print(conc)

    image_ind = image_path[-16:]
    #print(image_ind)

    find_row = conc[conc['Image Index'] == image_ind]
    #print(find_row)
    if find_row.empty:
        return "обьект не найден"

    find_row = find_row.iloc[:, -15:]
    find_row = np.array(find_row)
    #print(find_row)

    real_diseases = []
    for i in range(len(diseases)):
        if find_row[0][i] == 1:
            real_diseases.append(diseases[i])
    

    #print(type(find_row))
    return real_diseases


In [89]:
a = get_real_diseases('F:\\диплом\\3\\images_001\\images\\00000001_000.png')

In [90]:
a

'обьект не найден'

In [42]:
b = a.iloc[:, -15:]

In [43]:
b

,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
92170,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [44]:
c = np.array(b)

In [45]:
c

array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=int64)

In [46]:
c.shape

(1, 15)